# Download and run a rodent checkpoint

This notebook downloads a released rodent checkpoint and its reference data from MIMIC-MJX, restores the policy, generates one rollout, and renders it. Run it from the repository root or from `notebooks/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "glfw" if sys.platform == "darwin" else "egl"

import huggingface_hub as hf_hub
import mediapy as media

from track_mjx.agent import checkpointing
from track_mjx.analysis import rollout
from track_mjx.config import utils as config_utils

In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
checkpoint_id = "rodent/checkpoints-v1/feedforward_260210_013247_285744"
reference_file = "data/rodent/rodent_reference_clips.h5"
model_dir = repo_root / "model_checkpoints"

## Download the checkpoint and reference data

Existing files are reused from the local Hugging Face cache. The rodent reference file is large, so allow time and disk space for the first download.

In [ ]:
checkpoint_files = [
    filename
    for filename in hf_hub.HfApi().list_repo_files(
        repo_id="talmolab/MIMIC-MJX", repo_type="model"
    )
    if filename.startswith(f"{checkpoint_id}/")
]
if not checkpoint_files:
    raise FileNotFoundError(f"No released checkpoint found at {checkpoint_id}")
for filename in checkpoint_files:
    hf_hub.hf_hub_download(
        repo_id="talmolab/MIMIC-MJX",
        repo_type="model",
        filename=filename,
        local_dir=model_dir,
    )
reference_path = Path(
    hf_hub.hf_hub_download(
        repo_id="talmolab/MIMIC-MJX",
        repo_type="dataset",
        filename=reference_file,
        local_dir=repo_root,
    )
)
checkpoint_path = model_dir / checkpoint_id
print("Checkpoint:", checkpoint_path)
print("Reference data:", reference_path)

## Restore the policy and create the environment

In [ ]:
checkpoint = checkpointing.load_checkpoint_for_eval(checkpoint_path)
cfg = checkpoint["cfg"]
cfg.data_path = reference_path
cfg, cfg_dict, env_cfg_ml = config_utils.prepare_config(cfg)

env = rollout.create_environment(cfg.env_config)
inference_fn = checkpointing.load_inference_fn(cfg, checkpoint["policy"])
generate_rollout = rollout.create_rollout_generator(
    cfg,
    env,
    inference_fn,
    log_full_states=True,
    log_activations=False,
    log_metrics=False,
    log_sensor_data=False,
)

## Generate and render one rollout

The first execution includes JAX compilation and can take several minutes.

In [ ]:
single_rollout = generate_rollout(clip_idx=0)
frames = env.render_optimized(
    single_rollout,
    camera=f"{cfg.render_config.render_camera_name}{env._suffix}",
)
render_fps = round(1 / cfg.env_config.ctrl_dt)
output_path = checkpoint_path / "rollout.mp4"
media.write_video(output_path, frames, fps=render_fps)
media.show_video(frames, fps=render_fps)
print("Saved rollout to", output_path)